# Chapter 5: AI Gateway Demo

**From *AI Enterprise Architect***

Build a simple AI gateway that routes requests to different models based on task type,
logs all interactions, and enforces token budgets. This is the architectural pattern
that prevents AI sprawl and controls costs across the enterprise.

## What You'll Learn
- How to build a centralized AI gateway for model routing
- Task-based routing: match request types to appropriate (cost-effective) models
- Interaction logging for audit, debugging, and chargeback
- Token budget enforcement to control daily AI spend

## Prerequisites
- An OpenAI API key
- Basic Python knowledge

In [ ]:
# Install dependencies
!pip install -q openai

## 1. Configuration

Set your OpenAI API key as an environment variable. In Google Colab, you can use
the Secrets panel (key icon in the left sidebar) or set it directly:

```python
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
```

In [ ]:
import os
import time
import json
from datetime import datetime, date
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple

from openai import OpenAI

# Retrieve API key from environment
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError(
        "Please set the OPENAI_API_KEY environment variable. "
        "In Colab, use the Secrets panel or: os.environ['OPENAI_API_KEY'] = 'sk-...'"
    )

client = OpenAI(api_key=OPENAI_API_KEY)
print("Configuration loaded successfully.")

## 2. Define the Model Tier Map and Cost Table

The gateway routes requests to models based on task type. Each tier has an associated
cost per token (approximate, for budgeting purposes).

In [ ]:
# Model routing configuration.
# Maps task types to model tiers with associated cost-per-token estimates.

MODEL_TIERS = {
    "cheap": {
        "model": "gpt-4o-mini",
        "cost_per_input_token": 0.15 / 1_000_000,   # $0.15 per 1M input tokens
        "cost_per_output_token": 0.60 / 1_000_000,  # $0.60 per 1M output tokens
    },
    "mid": {
        "model": "gpt-4o",
        "cost_per_input_token": 2.50 / 1_000_000,
        "cost_per_output_token": 10.00 / 1_000_000,
    },
    "top": {
        "model": "o3-mini",
        "cost_per_input_token": 1.10 / 1_000_000,
        "cost_per_output_token": 4.40 / 1_000_000,
    },
}

# Task-type to tier routing rules.
TASK_ROUTING = {
    "summarize": "cheap",    # Summaries are routine — use the cheapest model.
    "translate": "cheap",    # Translation is well-handled by smaller models.
    "analyze": "mid",        # Analysis needs better reasoning.
    "classify": "mid",       # Classification benefits from a stronger model.
    "reason": "top",         # Complex reasoning tasks get the best model.
    "plan": "top",           # Strategic planning needs top-tier capability.
}

print("Model tiers and routing rules defined.")
print(f"\nRouting table:")
for task, tier in TASK_ROUTING.items():
    model = MODEL_TIERS[tier]["model"]
    print(f"  {task:<12} -> {tier:<6} -> {model}")

## 3. Interaction Log

Every request through the gateway is logged. This log supports:
- **Cost accounting**: chargeback to business units
- **Debugging**: trace failures back to specific requests
- **Compliance**: audit trail for regulated industries
- **Optimization**: identify which tasks consume the most budget

In [ ]:
@dataclass
class InteractionLog:
    """A single logged interaction through the AI gateway."""
    timestamp: str
    task_type: str
    tier: str
    model_used: str
    input_tokens: int
    output_tokens: int
    cost_usd: float
    latency_ms: float
    status: str  # "success", "budget_exceeded", "error"
    prompt_preview: str  # First 80 chars of the prompt for debugging.

print("InteractionLog dataclass defined.")

## 4. The AIGateway Class

This is the core component. It wraps the OpenAI API and adds:
- **`route_request`**: Determines which model to use based on task type.
- **`check_budget`**: Verifies the request won't exceed the daily budget.
- **`log_interaction`**: Records the full interaction for audit and analysis.
- **`send_request`**: The main entry point that orchestrates everything.

In [ ]:
class AIGateway:
    """
    Centralized AI gateway that routes requests, enforces budgets,
    and logs all interactions.
    """

    def __init__(
        self,
        client: OpenAI,
        daily_budget_usd: float = 1.00,
    ):
        self.client = client
        self.daily_budget_usd = daily_budget_usd
        self.logs: List[InteractionLog] = []
        self._daily_spend: float = 0.0
        self._budget_date: str = date.today().isoformat()

    # --- Routing ---
    def route_request(self, task_type: str) -> Tuple[str, str]:
        """
        Determine the model tier and model name for a given task type.
        Falls back to 'mid' tier for unknown task types.
        """
        tier = TASK_ROUTING.get(task_type, "mid")
        model = MODEL_TIERS[tier]["model"]
        return tier, model

    # --- Budget enforcement ---
    def check_budget(self, estimated_cost: float = 0.0) -> bool:
        """
        Check whether the daily budget allows another request.
        Resets the budget counter if the date has changed.
        Returns True if the request is allowed.
        """
        today = date.today().isoformat()
        if today != self._budget_date:
            # New day — reset the counter.
            self._daily_spend = 0.0
            self._budget_date = today

        return (self._daily_spend + estimated_cost) <= self.daily_budget_usd

    # --- Logging ---
    def log_interaction(self, log_entry: InteractionLog):
        """Append an interaction log entry."""
        self.logs.append(log_entry)

    # --- Main entry point ---
    def send_request(
        self,
        task_type: str,
        prompt: str,
        max_tokens: int = 200,
    ) -> Optional[str]:
        """
        Send a request through the gateway. Handles routing, budget
        checking, API call, logging, and error handling.

        Returns the model response text, or None if the request was rejected.
        """
        tier, model = self.route_request(task_type)

        # Pre-flight budget check (rough estimate).
        if not self.check_budget():
            log_entry = InteractionLog(
                timestamp=datetime.utcnow().isoformat(),
                task_type=task_type,
                tier=tier,
                model_used=model,
                input_tokens=0,
                output_tokens=0,
                cost_usd=0.0,
                latency_ms=0.0,
                status="budget_exceeded",
                prompt_preview=prompt[:80],
            )
            self.log_interaction(log_entry)
            print(f"  [BLOCKED] Budget exceeded. Daily spend: ${self._daily_spend:.4f} / ${self.daily_budget_usd:.2f}")
            return None

        # Make the API call.
        start_time = time.time()
        try:
            response = self.client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
            )
            latency_ms = (time.time() - start_time) * 1000

            # Extract usage.
            input_tokens = response.usage.prompt_tokens
            output_tokens = response.usage.completion_tokens
            result_text = response.choices[0].message.content

            # Calculate cost.
            tier_config = MODEL_TIERS[tier]
            cost = (
                input_tokens * tier_config["cost_per_input_token"]
                + output_tokens * tier_config["cost_per_output_token"]
            )
            self._daily_spend += cost

            # Log the interaction.
            log_entry = InteractionLog(
                timestamp=datetime.utcnow().isoformat(),
                task_type=task_type,
                tier=tier,
                model_used=model,
                input_tokens=input_tokens,
                output_tokens=output_tokens,
                cost_usd=cost,
                latency_ms=latency_ms,
                status="success",
                prompt_preview=prompt[:80],
            )
            self.log_interaction(log_entry)
            return result_text

        except Exception as e:
            latency_ms = (time.time() - start_time) * 1000
            log_entry = InteractionLog(
                timestamp=datetime.utcnow().isoformat(),
                task_type=task_type,
                tier=tier,
                model_used=model,
                input_tokens=0,
                output_tokens=0,
                cost_usd=0.0,
                latency_ms=latency_ms,
                status=f"error: {str(e)[:60]}",
                prompt_preview=prompt[:80],
            )
            self.log_interaction(log_entry)
            print(f"  [ERROR] {e}")
            return None

    # --- Reporting ---
    def budget_report(self) -> Dict:
        """Return a summary of budget utilization."""
        return {
            "daily_budget_usd": self.daily_budget_usd,
            "spent_usd": round(self._daily_spend, 6),
            "remaining_usd": round(self.daily_budget_usd - self._daily_spend, 6),
            "utilization_pct": round(self._daily_spend / self.daily_budget_usd * 100, 2),
            "total_requests": len(self.logs),
            "successful": sum(1 for l in self.logs if l.status == "success"),
            "blocked": sum(1 for l in self.logs if l.status == "budget_exceeded"),
        }


print("AIGateway class defined.")

## 5. Run 10 Sample Requests Through the Gateway

We send a mix of task types to exercise all model tiers. The gateway
automatically routes each request to the appropriate model.

In [ ]:
# Create the gateway with a modest daily budget.
gateway = AIGateway(client=client, daily_budget_usd=0.50)

# 10 sample requests of mixed types.
SAMPLE_REQUESTS = [
    ("summarize", "Summarize the key benefits of enterprise architecture in 2 sentences."),
    ("analyze", "Analyze the trade-offs between microservices and monolithic architectures for a mid-size bank."),
    ("summarize", "Summarize what TOGAF stands for and its primary purpose."),
    ("reason", "A company has 200 microservices, 15 teams, and no service mesh. They want to add AI capabilities. What architectural approach would you recommend and why?"),
    ("classify", "Classify this support ticket: 'My dashboard is loading slowly and the charts are not rendering.' Categories: performance, UI bug, data issue, access problem."),
    ("summarize", "Summarize the difference between data lakehouse and data warehouse in one paragraph."),
    ("translate", "Translate to French: 'The enterprise architecture review board meets every Thursday.'"),
    ("analyze", "Analyze the security implications of using a third-party LLM API for processing customer data."),
    ("reason", "Design a decision framework for choosing between build vs buy for AI capabilities in a regulated industry."),
    ("plan", "Create a 3-phase roadmap for migrating a legacy data warehouse to a modern lakehouse architecture."),
]

print(f"Sending {len(SAMPLE_REQUESTS)} requests through the AI gateway...\n")

for i, (task_type, prompt) in enumerate(SAMPLE_REQUESTS, 1):
    tier, model = gateway.route_request(task_type)
    print(f"Request {i:>2}/{len(SAMPLE_REQUESTS)}: [{task_type:<10}] -> {model}")

    result = gateway.send_request(task_type, prompt, max_tokens=150)

    if result:
        # Show a truncated preview of the response.
        preview = result[:150].replace("\n", " ")
        print(f"           Response: {preview}...")
    print()

print("All requests processed.")

## 6. Display the Interaction Log

The full interaction log shows every request, its routing decision, cost, and latency.
This is the data you would feed into a dashboard for AI governance.

In [ ]:
# Display the interaction log as a formatted table.
print(f"{'#':<3} {'Timestamp':<26} {'Task':<11} {'Tier':<6} {'Model':<14} "
      f"{'In Tok':>7} {'Out Tok':>8} {'Cost ($)':>9} {'Latency':>9} {'Status':<10}")
print("-" * 115)

for i, log in enumerate(gateway.logs, 1):
    print(
        f"{i:<3} {log.timestamp:<26} {log.task_type:<11} {log.tier:<6} "
        f"{log.model_used:<14} {log.input_tokens:>7} {log.output_tokens:>8} "
        f"{log.cost_usd:>9.6f} {log.latency_ms:>8.0f}ms {log.status:<10}"
    )

## 7. Budget Utilization and Cost Breakdown

Analyze spending by model tier. This is the data a FinOps team would use to
optimize AI costs.

In [ ]:
# Budget report
report = gateway.budget_report()
print("=== Budget Report ===")
for key, value in report.items():
    print(f"  {key:<20}: {value}")

# Cost breakdown by tier
print("\n=== Cost Breakdown by Model Tier ===")
tier_costs: Dict[str, Dict] = {}
for log in gateway.logs:
    if log.status != "success":
        continue
    if log.tier not in tier_costs:
        tier_costs[log.tier] = {
            "requests": 0,
            "total_cost": 0.0,
            "total_input_tokens": 0,
            "total_output_tokens": 0,
            "avg_latency_ms": [],
        }
    tier_costs[log.tier]["requests"] += 1
    tier_costs[log.tier]["total_cost"] += log.cost_usd
    tier_costs[log.tier]["total_input_tokens"] += log.input_tokens
    tier_costs[log.tier]["total_output_tokens"] += log.output_tokens
    tier_costs[log.tier]["avg_latency_ms"].append(log.latency_ms)

print(f"\n{'Tier':<8} {'Model':<14} {'Reqs':>5} {'Cost ($)':>10} "
      f"{'In Tok':>8} {'Out Tok':>9} {'Avg Lat':>9}")
print("-" * 70)
for tier, data in tier_costs.items():
    avg_lat = sum(data["avg_latency_ms"]) / len(data["avg_latency_ms"])
    model = MODEL_TIERS[tier]["model"]
    print(
        f"{tier:<8} {model:<14} {data['requests']:>5} "
        f"{data['total_cost']:>10.6f} {data['total_input_tokens']:>8} "
        f"{data['total_output_tokens']:>9} {avg_lat:>8.0f}ms"
    )

# Total cost
total = sum(d["total_cost"] for d in tier_costs.values())
print(f"\nTotal spend: ${total:.6f}")
print(f"Budget remaining: ${report['remaining_usd']:.6f} ({100 - report['utilization_pct']:.1f}%)")

## 8. Test Budget Enforcement

Let's lower the budget to see the gateway reject requests when the budget is exhausted.

In [ ]:
# Create a very low-budget gateway to demonstrate budget enforcement.
tight_gateway = AIGateway(client=client, daily_budget_usd=0.001)  # $0.001 budget

print("Testing budget enforcement with a $0.001 daily budget...\n")

for i in range(5):
    result = tight_gateway.send_request(
        "summarize",
        f"Summarize what enterprise architecture is in one sentence. (Request {i+1})",
        max_tokens=50,
    )
    status = tight_gateway.logs[-1].status
    spend = tight_gateway._daily_spend
    print(f"  Request {i+1}: status={status}, cumulative_spend=${spend:.6f}")

print(f"\nFinal budget report:")
for k, v in tight_gateway.budget_report().items():
    print(f"  {k}: {v}")

## Key Takeaways

1. **Centralized AI governance prevents sprawl.** Without a gateway, every team
   independently calls whatever model they want, with no visibility or control.

2. **Task-based routing optimizes cost.** Not every request needs the most expensive
   model. Routing summaries to a cheap model and reasoning to a top-tier model can
   reduce costs by 70-90%.

3. **Interaction logging enables FinOps.** You can charge back AI costs to business
   units, identify optimization opportunities, and maintain compliance audit trails.

4. **Budget enforcement is a safety net.** Runaway scripts or prompt injection attacks
   can generate thousands of API calls. A budget gate prevents surprise bills.

### For Enterprise Architects

The AI gateway is an architectural pattern, not just a piece of code. In production,
you would deploy it as a shared service (API or sidecar), integrate it with your
identity provider for per-team budgets, and feed the logs into your observability
stack (Datadog, Grafana, Splunk). Treat it like your API gateway — but for AI.